In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip  # se tiverem delta instalado via pip

builder = (
    SparkSession.builder
    .appName("Spark_DFs_SQL_Tourism")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")  # se for preciso trazer o jar
    .enableHiveSupport()
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession com Delta inicializada!")
print("Versão do Spark:", spark.version)

✔️ SparkSession com Delta inicializada!
Versão do Spark: 3.4.1


In [2]:
from pyspark.sql.functions import col

DELTA_PATH = "hdfs://hdfs-nn:9000/demo/silver/boxoffice_silver/"

df_silver = (
    spark.read
        .format("delta")
        .load(DELTA_PATH)
)

df_silver.printSchema()
df_silver.show(20, truncate=False)

root
 |-- Year: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- gross_clean: long (nullable = true)

+----+-----------------------------------------+-----------+
|Year|Title                                    |gross_clean|
+----+-----------------------------------------+-----------+
|2007|Pirates of the Caribbean: At World's End |960996492  |
|2007|Harry Potter and the Order of the Phoenix|938212738  |
|2007|Spider-Man 3                             |894983373  |
|2007|Shrek the Third                          |813367380  |
|2007|Transformers                             |709709780  |
|2007|Ratatouille                              |623726085  |
|2007|I Am Legend                              |585410052  |
|2007|The Simpsons Movie                       |536414270  |
|2007|National Treasure: Book of Secrets       |459242249  |
|2007|300                                      |456068181  |
|2007|The Bourne Ultimatum                     |444100035  |
|2007|Live Free or Die Ha

In [7]:
df_silver.createOrReplaceTempView("boxoffice_silver")

spark.sql("""
    SELECT 
        Title,
        Year,
        gross_clean
    FROM boxoffice_silver
    Where Year = 2010
    ORDER BY gross_clean DESC
    LIMIT 10
""").show(truncate=False)

+--------------------------------------------+----+-----------+
|Title                                       |Year|gross_clean|
+--------------------------------------------+----+-----------+
|Toy Story 3                                 |2010|1066969703 |
|Alice in Wonderland                         |2010|1025467110 |
|Harry Potter and the Deathly Hallows: Part 1|2010|960283305  |
|Inception                                   |2010|828258695  |
|Shrek Forever After                         |2010|752600867  |
|The Twilight Saga: Eclipse                  |2010|698491347  |
|Iron Man 2                                  |2010|623933331  |
|Tangled                                     |2010|592461732  |
|Despicable Me                               |2010|543239815  |
|How to Train Your Dragon                    |2010|494878759  |
+--------------------------------------------+----+-----------+



As you can see, you now have a Spark DataFrame created from your JSON file in HDFS :)

You can now use the Spark DataFrame functions to interact with DataFrames: Spark Dataframe operations

A complete list of DataFrame operations with examples can be found here https://spark.apache.org/docs/3.4.1/api/python/reference/pyspark.sql/core_classes.html. There you can find how to apply filter and group functions, joins, iterate through the dataframe and apply python functions, drop dupicates, among many other operations at the DataFrame level.

A complete list of Spark SQL functions with examples to apply to the columns of DataFrames can be found here https://spark.apache.org/docs/latest/api/sql/index.html. There you can find how to check for nulls, do string replaces, among many others.

In [4]:
df_silver.createOrReplaceTempView("tourism_silver")

In [5]:
# In case we want to do complementary analysis to find out distinct values for particular columns, being used for filtering
movies_silver.printSchema()
movies_silver.show(10, truncate=False)

movies_silver.groupBy("release_year").count().orderBy("release_year").show()

NameError: name 'movies_silver' is not defined

In [ ]:
hdfs_path_silver = "hdfs://hdfs-nn:9000/demo/silver/movies_silver"

(
    movies_silver
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("release_year")
        .save(hdfs_path_silver)
)